# {{PROJECT_NAME}}

KFP project on the Miramar platform.

- Define pipeline components in the tagged cells below (`kfp_step`, `kfp_pipeline`)
- Run the **Build → `pipeline.py`** cell to regenerate `pipeline.py`
- Use the Compile & Submit cells to test runs locally
- Use **Deploy to KFP** (GitHub Actions) for CI/CD submission

## Pipeline definition

Define your components below. Tag each component cell `kfp_step` and the pipeline cell `kfp_pipeline`.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=["nvtx"],
)
def gpu_stage(run_id: str):
    import torch
    import nvtx

    # ── Replace this block with your GPU workload ─────────────────────────
    with nvtx.annotate("gpu_stage", color="green"):
        x = torch.randn(2048, 2048, device="cuda")
        result = x @ x.T
        torch.cuda.synchronize()

    print(f"Result norm: {result.norm().item():.4f}")
    # ─────────────────────────────────────────────────────────────────────

In [ ]:
@dsl.pipeline(
    name="{{PROJECT_NAME}}",
    description="GPU pipeline — replace gpu_stage with your workload.",
)
def pipeline(run_id: str = "run-001"):
    task = gpu_stage(run_id=run_id)
    task.set_gpu_limit(1).set_memory_limit("64G")
    # To profile with Nsight Operator:
    # kubernetes.add_pod_label(task, label_key="nvidia-nsight-profile", label_value="enabled")

## Build → `pipeline.py`

Save the notebook first (`Ctrl+S`), then run this cell to regenerate `pipeline.py`.

In [ ]:
import json, pathlib, re


def build_pipeline(notebook_path="notebook.ipynb"):
    nb = json.loads(pathlib.Path(notebook_path).read_text())
    step_srcs, pipeline_src = [], None
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        tags = cell.get("metadata", {}).get("tags", [])
        src = "".join(cell["source"])
        if "kfp_step" in tags:
            step_srcs.append(src)
        elif "kfp_pipeline" in tags:
            pipeline_src = src
    if not step_srcs:
        raise RuntimeError("No cells tagged 'kfp_step' found.")
    if pipeline_src is None:
        raise RuntimeError("No cell tagged 'kfp_pipeline' found.")
    names = [
        m.group(1)
        for src in step_srcs
        for m in [re.search(r"^def (\w+)\(", src, re.MULTILINE)]
        if m
    ]
    out  = "# Generated by notebook.ipynb — do not edit manually.\n"
    out += "# Re-run the Build cell to regenerate.\n\n"
    out += "from kfp import dsl\n"
    out += "from kfp import kubernetes\n\n\n"
    out += "\n\n\n".join(step_srcs)
    out += "\n\n\n"
    out += pipeline_src
    out += "\n"
    pathlib.Path("pipeline.py").write_text(out)
    print(f"Wrote pipeline.py — {len(names)} component(s): {', '.join(names)}")


build_pipeline()

## Compile & Submit

Run these cells to compile and submit directly from Jupyter (useful for quick iteration).

In [ ]:
import kfp
from kfp import compiler, dsl
print(f'kfp {kfp.__version__}')

In [ ]:
# Compile the pipeline defined in pipeline.py
from pipeline import pipeline
compiler.Compiler().compile(pipeline_func=pipeline, package_path='/tmp/pipeline.yaml')
print('Compiled → /tmp/pipeline.yaml')

In [ ]:
# Connect to KFP (requires SSH tunnel: ssh -L 8080:localhost:8080 spark-79b7.local)
client = kfp.Client(host='http://localhost:8080')
client.list_pipelines()

In [ ]:
# Submit a run
run = client.create_run_from_pipeline_package(
    pipeline_file='/tmp/pipeline.yaml',
    arguments={},
    run_name='notebook-run',
)
print(f'Run ID: {run.run_id}')
print(f'UI: http://localhost:8080/#/runs/details/{run.run_id}')

In [ ]:
# Poll run status
import time
run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.state
    print(f'  {state}')
    if state in ('SUCCEEDED', 'FAILED', 'CANCELED'):
        break
    time.sleep(10)